# Tutorial 05: Extending PhenoMe (Plugins and Models)

**About this tutorial**
- **Time:** ~45 minutes
- **Prerequisites:** [Tutorial 03](03_core_phenotyping_workflow.ipynb)
- **Key ideas:** Property plugins, custom `ModelWrapper`, registries

This tutorial is for users who want to push the boundaries: adding custom math formulas to measure highly specific cell traits, or swapping out the underlying AI model for a completely different one.

1. **Custom Property Plugins**: Writing your own logical rules to extract physical features.
2. **Custom AI Models**: Using a different Base AI Model instead of the built-in DINOv2.

## 1. Custom Property Plugins

If you need to measure something highly specific to your experiment like counting the exact number of bright spots inside a cell, or measuring highly specific texture rules you can write a mini-script (called a plugin) to do it.

PhenoMe provides a **plugin registry** that allows you to easily name, save, and reuse these custom formulas across different experiments.

In [ ]:
import numpy as np
from phenome import PhenoMe
from phenome.plugins import register_property, get_blob_properties

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pheno = PhenoMe(device=device, seed=42)

# 1. Define a custom property function 
# It must accept (image2d, mask2d) and return a dictionary of {name: value}
def my_peak_intensity(image2d, mask2d):
    """Simple extractor for the peak intensity in an image."""
    if image2d is None: 
        return {"peak_val": np.nan}
    return {"peak_val": float(np.max(image2d))}

# 2. Register it globally in the PhenoMe plugin system
register_property("peak_intensity", my_peak_intensity)

# 3. Use it during compute_properties
pheno.compute_properties(
    property_preset="basic",
    additional_property_functions={
        "custom_peak": my_peak_intensity,        # Direct function
        "auto_blobs": get_blob_properties       # Built-in plugin
    }
)

## 2. Custom Base AI Models

PhenoMe uses the highly advanced DINOv2 model by default to observe your cells. However, if your research requires an entirely different approach, you can swap it out. Whether you want to plug in ResNet, CLIP, or your own custom-trained model, you simply wrap it in a `ModelWrapper` and let the pipeline do the rest.

In [ ]:
from phenome.models import ModelWrapper
import torch
from torchvision.models import resnet18, resnet50, ResNet50_Weights

# Example 1: Simple ResNet18 wrapper
class MyResNetWrapper(ModelWrapper):
    def _get_embeddings(self, tensor: torch.Tensor) -> torch.Tensor:
        # Custom logic to extract embeddings from the model output
        out = self.model(tensor)
        if out.ndim == 4:
            out = out.flatten(1)
        return out

# model = resnet18(pretrained=True)
# wrapper = MyResNetWrapper(model)
# pheno.process_images(wrapper)

# Example 2: More complex setup with torchvision model
# model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
# model = torch.nn.Sequential(*list(model.children())[:-1]) # Remove classification head
# wrapper = MyResNetWrapper(model)

## See Also

- [03_core_phenotyping_workflow.ipynb](03_core_phenotyping_workflow.ipynb): Core API tutorial
- [04_exploratory_analysis_and_explainability.ipynb](04_exploratory_analysis_and_explainability.ipynb): Advanced clustering and explainability